In [2]:
# -----######-----###### COPY SONGS INTO BATCHES w/ AIFF + WAV FIRST -----######-----######
import shutil
from pathlib import Path
import pandas as pd
from tqdm import tqdm

def _songs_0108_batches_GET_organized(txt_path, output_root, batch_size=30):
    tqdm.pandas()

    # ---------- Read + Clean ----------
    df = pd.read_csv(
        txt_path,
        sep='\t',
        encoding='utf-16',
        engine='python',
        on_bad_lines='skip'
    )
    df.columns = df.columns.str.strip()
    df = df.dropna(subset=['Location'])
    df['Location'] = df['Location'].astype(str).str.strip().str.replace('\u200b', '', regex=False)

    # ---------- Prioritize AIFF & WAV ----------
    df['ext'] = df['Location'].apply(lambda x: Path(x).suffix.lower())
    ext_priority = ['.aiff', '.wav']
    df['sort_key'] = df['ext'].apply(lambda x: 0 if x in ext_priority else 1)
    df = df.sort_values(by='sort_key').reset_index(drop=True)

    # ---------- Setup Copy ----------
    seen_files = set()
    batch_counter = 1
    file_counter = 0
    output_root = Path(output_root)
    current_batch = output_root / f"_batch_{batch_counter}"
    current_batch.mkdir(parents=True, exist_ok=True)

    for i, path_str in enumerate(tqdm(df['Location'], desc="📂 Copying songs")):
        raw_path = str(path_str).strip().replace('\u200b', '').replace('\xa0', '').replace('\r', '')
        source = Path(raw_path)
        file_name = source.name

        if not source.is_file():
            print(f"❌ Missing: {source}")
            continue

        # Move to next batch if duplicate or batch full
        if file_name in seen_files or file_counter >= batch_size:
            batch_counter += 1
            file_counter = 0
            current_batch = output_root / f"_batch_{batch_counter}"
            current_batch.mkdir(parents=True, exist_ok=True)

        target = current_batch / file_name

        if not target.exists():
            try:
                shutil.copy2(source, target)
                seen_files.add(file_name)
                file_counter += 1
            except Exception as e:
                print(f"❌ Error copying {source}: {e}")


In [3]:
txt_path = "/Users/yerik/Desktop/arch_now.txt"
output_dir = "/Volumes/_1_HD_YODJ/_0_ARCH_2022-July-2025"

_songs_0108_batches_GET_organized(txt_path, output_dir, batch_size=30)


📂 Copying songs:  68%|██████████████████████████████████▉                | 1836/2684 [16:13<02:53,  4.90it/s]

❌ Missing: /Users/yerik/Music/_0_OLD_SOURCE/__tiny_12_07-25/Yaeji feat. YonYon, G.L.A.M. - SPELL주문 myfreemp3.vip .mp3


📂 Copying songs:  80%|████████████████████████████████████████▌          | 2134/2684 [16:42<00:41, 13.37it/s]

❌ Missing: /Users/yerik/Music/_0_OLD_SOURCE/__tiny_34_07-25/Bad Bunny Ft. El Alfa - La Romana (Fuego)- DjVivaEdit Dembow Drop In+Intro+Outro.mp3


📂 Copying songs: 100%|███████████████████████████████████████████████████| 2684/2684 [17:37<00:00,  2.54it/s]
